# 03. Experimentación y Selección de Modelos

Con los datos limpios, enriquecidos y escalados, es hora de encontrar el algoritmo predictivo ganador.

### Instrucciones Generales:
1. **Validación:** No entrenes y midas sobre el mismo conjunto (sobreajuste). Recuerda haber dividido en Entrenamiento y Prueba antes.
2. **Entrenamiento Base:** Entrena los siguientes modelos base con tu set de Entrenamiento y compáralos usando RMSE (Error Cuadrático Medio):
   - `LinearRegression`
   - `SGDRegressor`
   - `DecisionTreeRegressor`
   - `RandomForestRegressor`
3. **Cross Validation (Validación Cruzada):** Para tener una métrica robusta, usa `cross_val_score` en el set de Entrenamiento para cada uno de los modelos anteriores.
4. **Ajuste Fino (Fine Tuning):** Toma el modelo ganador y busca sus mejores hiperparámetros. Utiliza un `GridSearchCV` explorando el número de estimadores (`n_estimators`), las características máximas (`max_features`), etc.
5. **Conclusión y Benchmark (IMPORTANTE):** Redacta una conclusión comparando los algoritmos. Explica por qué escogiste el modelo final y valida tu decisión calculando el RMSE sobre tu Set de Prueba que habías reservado. Documenta si alguno de tus modelos se sobreajusto o subajusto. Recuerda que el modelo final no puede tener esos problemas!


In [19]:
# Empieza importando los algoritmos desde Scikit-Learn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler,RobustScaler
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [20]:
# 1. Carga de datos
df = pd.read_csv('clean_data.csv')

# 2. Preparación de X (características) y y (objetivo)
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

# Escalamiento de datos
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# 3. División en entrenamiento y validación (80-20)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# 4. Entrenamiento y Comparación
models = {
    "LinearRegression": LinearRegression(),
    "SGDRegressor": SGDRegressor(max_iter=1000, tol=1e-3, random_state=42),
    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=42),
    "RandomForestRegressor": RandomForestRegressor(random_state=42)
}

In [21]:
df.describe()

,longitude,latitude,total_rooms,total_bedrooms,population,households,median_house_value,median_income_log,ocean_ordinal,age_ordinal,rooms_per_household,bedrooms_per_room,population_per_household,income_per_person
count,13670.000000,13670.000000,13670.000000,13670.000000,13670.000000,13670.000000,13670.000000,13670.000000,13670.000000,13670.000000,13670.00000,13670.000000,13670.000000,13670.000000
mean,-119.613619,35.711342,2080.028456,425.530358,1147.382882,398.412143,189885.515435,1.484098,1.002853,1.098830,5.37588,0.213164,2.963623,1.315920
std,2.001957,2.166649,1026.400800,199.710611,549.163389,185.323077,98272.671800,0.333718,0.948637,0.741794,2.33353,0.055069,1.078400,0.627353
min,-124.350000,32.550000,2.000000,2.000000,3.000000,2.000000,14999.000000,0.405398,0.000000,0.000000,0.89000,0.100000,0.690000,0.025500
25%,-121.790000,33.940000,1353.000000,282.000000,753.000000,267.000000,112500.000000,1.250790,0.000000,1.000000,4.46000,0.177400,2.460000,0.839975
50%,-118.700000,34.410000,1937.000000,400.000000,1082.000000,378.000000,170800.000000,1.491319,1.000000,1.000000,5.21000,0.203600,2.840000,1.249200
75%,-118.020000,37.750000,2697.750000,553.000000,1502.000000,518.000000,244400.000000,1.718579,1.000000,2.000000,5.97000,0.238000,3.300000,1.689650
max,-114.490000,41.950000,5650.000000,1047.000000,2747.000000,905.000000,500000.000000,2.772595,4.000000,2.000000,132.53000,1.000000,63.750000,6.756800


In [23]:
import math
resultados_dict = {}

print(f"{'Modelo':<25} | {'RMSE Valid':<12} | {'RMSE Train':<12}| {'CV RMSE':<12}")
print("-" * 67)

for nombre, model in models.items():
    # Entrenamiento
    model.fit(X_train, y_train)
    
    # Predicciones
    y_train_pred = model.predict(X_train)
    y_valid_pred = model.predict(X_valid)
    
    # Cálculo de RMSE
    rmse_train = math.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_valid = math.sqrt(mean_squared_error(y_valid, y_valid_pred))
    
    # Cross Validation (usamos el set de entrenamiento completo)
    # cv=10 es un estándar robusto
    scores = cross_val_score(model, X_train, y_train, 
                             scoring="neg_mean_squared_error", cv=10)
    rmse_cv_scores = np.sqrt(-scores)
    
    # Guardar en el diccionario solicitado
    resultados_dict[nombre] = {
        'modelo nombre': nombre,
        'modelo': model,
        'rmse_train': rmse_train,
        'rmse_valid': rmse_valid,
        'rmse_cv_mean': rmse_cv_scores.mean()
    }
    
    print(f"{nombre:<25} | {rmse_valid:<12.2f} | | {rmse_train:<12.2f} | {rmse_cv_scores.mean():<12.2f}")

Modelo                    | RMSE Valid   | RMSE Train  | CV RMSE     
-------------------------------------------------------------------
LinearRegression          | 59718.13     | | 58958.69     | 59155.95    
SGDRegressor              | 59857.92     | | 59133.20     | 59270.26    
DecisionTreeRegressor     | 61499.71     | | 0.00         | 61161.59    
RandomForestRegressor     | 44461.06     | | 16705.20     | 45078.68    


from sklearn.model_selection import GridSearchCV


# Configuración y ejecución del GridSearch
param_grid = [
    {'n_estimators': [30, 100], 'max_features': [2, 4, 6, 8]},
    {'bootstrap': [False], 'n_estimators': [10, 20], 'max_features': [2, 3, 4]},
]

forest_reg = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
                           scoring='neg_mean_squared_error',
                           return_train_score=True)

grid_search.fit(X_train, y_train)

# Extraer el mejor modelo
best_model = grid_search.best_estimator_

# RMSE para Train y Valid con el MEJOR modelo
y_train_pred = best_model.predict(X_train)
y_valid_pred = best_model.predict(X_valid)

rmse_train_final = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_valid_final = np.sqrt(mean_squared_error(y_valid, y_valid_pred))

# Detalle
print(f"Mejores Hiperparámetros: {grid_search.best_params_}")
print("-" * 40)
print(f"{'Métrica':<20} | {'Valor (RMSE)':<15}")
print("-" * 40)
print(f"{'Best Model Train':<20} | {rmse_train_final:<15.2f}")
print(f"{'Best Model Valid':<20} | {rmse_valid_final:<15.2f}")
print("-" * 40)

# Resultado
mejor_resultado = {
    'modelo_nombre': 'Optimized RandomForest',
    'mejor_modelo': best_model,
    'rmse_train': rmse_train_final,
    'rmse_valid': rmse_valid_final,
    'best_params': grid_search.best_params_
}

In [24]:
from sklearn.model_selection import GridSearchCV
# Definimos la rejilla de parámetros para RandomForest
param_grid = [
    {
        'n_estimators': [100, 200], 
        'max_features': [4, 6, 8],
        'max_depth': [10, 20, None],       # None permite profundidad total, 10/20 la limitan
        'min_samples_split': [2, 5, 10],   # Mínimo de muestras para dividir un nodo
        'min_samples_leaf': [1, 2, 4],     # Mínimo de muestras en cada hoja
        'bootstrap': [True,False]
    }
]

forest_reg = RandomForestRegressor(random_state=42)

# Configuramos la búsqueda con validación cruzada (cv=5)
grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
                           scoring='neg_mean_squared_error'
                           )

grid_search.fit(X_train, y_train)

# Mejores parámetros y mejor estimador
print(f"Mejores parámetros: {grid_search.best_params_}")
rmse_error_train = math.sqrt(grid_search.best_score_ * -1)
final_model = grid_search.best_estimator_
valid_pred = final_model.predict(X_valid)
rmse_error_valid = math.sqrt(mean_squared_error(y_valid, valid_pred))
print(f'RMSE train Bosque: {rmse_error_train}') 
print(f'RMSE valid Bosque: {rmse_error_valid}')


Mejores parámetros: {'bootstrap': False, 'max_depth': None, 'max_features': 8, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
RMSE train Bosque: 45483.61287192449
RMSE valid Bosque: 44293.01499570674


In [25]:
# Guardar modelo
import joblib

# Guardamos el modelo y el escalador (asumiendo que se llama 'scaler')
joblib.dump(final_model, 'housing_model.joblib')
joblib.dump(scaler, 'scaler.joblib')

print("Modelo y escalador guardados con éxito.")

Modelo y escalador guardados con éxito.


In [26]:
model = joblib.load('housing_model.joblib')
scaler = joblib.load('scaler.joblib')
test_data = pd.read_csv(r'../data/interim/test_set.csv') # Asegúrate que este archivo exista

# Copia para no modificar el original
df_test = test_data.copy()

# --- Codificaciones Categóricas ---
ocean_mapping = {'INLAND': 0, '<1H OCEAN': 1, 'NEAR OCEAN': 2, 'NEAR BAY': 3, 'ISLAND': 4}
df_test['ocean_ordinal'] = df_test['ocean_proximity'].map(ocean_mapping).astype(int)

# --- Codificación de antiguedad ---
bins_age = [0, 18, 35, np.inf]
labels_age = ['Nueva', 'Media', 'Vieja']
age_mapping = {'Nueva': 0, 'Media': 1, 'Vieja': 2}
df_test['age_cat'] = pd.cut(df_test['housing_median_age'], bins=bins_age, labels=labels_age)
df_test['age_ordinal'] = df_test['age_cat'].map(age_mapping).astype(int)

# --- Ingeniería de Variables ---
df_test['rooms_per_household'] = df_test['total_rooms'] / df_test['households']
df_test['bedrooms_per_room'] = df_test['total_bedrooms'] / df_test['total_rooms']
df_test['population_per_household'] = df_test['population'] / df_test['households']
df_test['income_per_person'] = df_test['median_income'] / df_test['population']

# --- Logaritmo y limpieza de columnas ---
df_test['median_income_log'] = np.log1p(df_test['median_income'])

# --- Definir el orden de las 13 columnas de entrada (X) que espera el modelo ---
cols_input = [
    'longitude', 'latitude', 'total_rooms', 'total_bedrooms', 'population', 
    'households', 'median_income_log', 'ocean_ordinal', 'age_ordinal', 
    'rooms_per_household', 'bedrooms_per_room', 'population_per_household', 'income_per_person'
]

X_test = df_test[cols_input]
y_test = df_test['median_house_value']

# --- Escalamiento y Predicción ---
X_test_scaled = scaler.transform(X_test)
predictions = model.predict(X_test_scaled)

# Cálculo de RMSE
final_rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"RMSE en el conjunto de Test: ${final_rmse:,.2f}")

RMSE en el conjunto de Test: $101,879.98


#### Otro enfoque

In [11]:
# 1. Cargar los datos preparados en el cuaderno anterior
train_data = pd.read_csv('data_train_ML.csv')
val_data = pd.read_csv('data_valid_ML.csv')

# Asumiendo que 'median_house_value' es la columna objetivo (Target)
X_train = train_data.drop('median_house_value', axis=1)
y_train = train_data['median_house_value']

X_val = val_data.drop('median_house_value', axis=1)
y_val = val_data['median_house_value']

# 2. Definir los modelos
models = {
    "RegresionLineal": LinearRegression(), # No tiene hiperparámetros de tuning significativos
    
    "SGDRegressor": SGDRegressor(
        loss='squared_error',
        penalty='l2',              # Regularización para evitar pesos gigantes
        alpha=0.001,
        learning_rate='adaptive',  # Reduce el paso de aprendizaje si el error no baja
        eta0=0.001,                 # Tasa de aprendizaje inicial
        max_iter=10000, 
        random_state=42
    ),
    
    "DecisionTreeRegressor": DecisionTreeRegressor(
        max_depth=10,              # Un árbol muy profundo dispara el RMSE en validación
        min_samples_leaf=50,       # Suaviza las predicciones en los extremos
        min_samples_split=20,  # Suele dar mejores resultados en regresión
        random_state=42
    ),
    
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=300,          # Más árboles reducen la varianza del error
        max_depth=15,              # Suficiente profundidad para segmentar latitud/longitud
        min_samples_split=10,       # Evita crear ramas basadas en ruidos/outliers
        min_samples_leaf=20,       # Obliga a los árboles a ser diversos
        bootstrap=True,
        random_state=42
    )
}

# Diccionario final para guardar (nombre, objeto, rmse_train, rmse_valid)
results_dict = {}

print(f"{'Modelo':<25} | {'RMSE Train (CV)':<15} | {'RMSE Validation':<15}")
print("-" * 60)

for name, model in models.items():
    # A. Validación Cruzada en Entrenamiento (RMSE robusto)
    # Negamos el score porque cross_val_score busca maximizar, y el MSE es mejor si es menor
    scores = cross_val_score(model, X_train, y_train, 
                             scoring="neg_mean_squared_error", cv=10)
    rmse_train_cv = np.sqrt(-scores.mean())
    
    # B. Ajuste final y evaluación en Validación
    model.fit(X_train, y_train)
    y_pred_val = model.predict(X_val)
    rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
    
    # C. Guardar en el diccionario solicitado
    results_dict = {
        "nombre_modelo": name,
        "modelo_obj": model,
        "rmse_train": rmse_train_cv,
        "rmse_valid": rmse_val
    }
    
    print(f"{name:<25} | {rmse_train_cv:>15.2f} | {rmse_val:>15.2f}")

Modelo                    | RMSE Train (CV) | RMSE Validation
------------------------------------------------------------
RegresionLineal           |        58935.53 |        60611.36


c:\Users\DETPC\anaconda3\envs\mi_entorno1\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


SGDRegressor              |   2169315506.55 |   2728214301.79
DecisionTreeRegressor     |        52783.09 |        54957.46
RandomForestRegressor     |        47889.98 |        49422.30


### Benchmark y Conclusión Final
*(Escribe tus conclusiones de negocio aquí)*